# 05 — Clustering me K-Means
**Dataset:** Credit Card Fraud Detection — të dhëna të parapërpunuara nga `02_preprocessing.ipynb`

Qëllimi: Zbulojmë struktura natyrore në të dhëna **pa përdorur labels** (unsupervised learning). Krahasojmë clusterat e gjetur me klasat reale për të vlerësuar sa mirë K-Means ndan transaksionet mashtruese nga ato legjitime.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

RANDOM_STATE = 42

In [ ]:
X_all         = np.load('../data/processed/X_all.npy')
y_all         = np.load('../data/processed/y_all.npy')
feature_names = np.load('../data/processed/feature_names.npy', allow_pickle=True)

print(f'X_all shape    : {X_all.shape}')
print(f'y_all shape    : {y_all.shape}')
print(f'Features ({len(feature_names)}): {list(feature_names)}')
print(f'\nKlasa 0 (legjitime) : {(y_all == 0).sum():,}')
print(f'Klasa 1 (mashtruese): {(y_all == 1).sum():,}')
print(f'\nLabels NUK përdoren gjatë clustering — unsupervised!')

## 1. Zgjedhja e K Optimal — Elbow Method
Testojmë K nga 2 deri 10 dhe shikojmë kur inertia (shuma e distancave brenda clusterit) ndalon së uluri ndjeshëm — "bërryli" i grafikut tregon K optimal.

In [ ]:
K_range  = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_all)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_all, km.labels_, sample_size=10000, random_state=RANDOM_STATE)
    silhouettes.append(sil)
    print(f'K={k}  |  Inertia: {km.inertia_:,.0f}  |  Silhouette: {sil:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_title('Elbow Method — Inertia sipas K', fontweight='bold')
axes[0].set_xlabel('Numri i Clusterave (K)')
axes[0].set_ylabel('Inertia')
axes[0].axvline(x=2, color='tomato', linestyle='--', linewidth=1.5, label='K optimal')
axes[0].legend()

axes[1].plot(list(K_range), silhouettes, 'gs-', linewidth=2, markersize=8)
axes[1].set_title('Silhouette Score sipas K', fontweight='bold')
axes[1].set_xlabel('Numri i Clusterave (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].axvline(x=2, color='tomato', linestyle='--', linewidth=1.5, label='K optimal')
axes[1].legend()

plt.suptitle('Zgjedhja e K Optimal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nK=2 zgjidhet: dataset ka 2 klasa natyrore (legjitime / mashtruese)')